In [1]:
from utils.experiment_utils import get_all_experiments_info, load_best_model
import torch
import os
import hydra
from omegaconf import DictConfig, OmegaConf

import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader

import ot

from datasets.lineage_tracing import LTSeqDataset

from geomloss import SamplesLoss

from sklearn.model_selection import train_test_split
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error, r2_score

from utils.eval_utils import compute_mmd_distance, compute_sw_distance

from sklearn.neighbors import NearestNeighbors

import ot

/orcd/home/002/gokulg/miniforge3/envs/gde/lib/python3.11/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/orcd/home/002/gokulg/miniforge3/envs/gde/lib/python3.11/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/orcd/home/002/gokulg/miniforge3/envs/gde/lib/python3.11/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


In [2]:
lts = LTSeqDataset(seed=42)

loading cached adata from ./data/processed/adata_pca_50.h5ad  !!
loading cached clone sets from ./data/processed!
splitting 1218 clones into 609 train and 609 test


In [3]:
# lineageOT baseline
# compute OT coupling between early/late clones by mean states
# then "lift" to new data points by nearest neighbor

results = {
    'energy' : [],
    'mmd' : [],
    'swd' : []
}

early = lts.train_srcs.mean(dim=1).numpy()
late = lts.train_tgts.mean(dim=1).numpy()

C = ot.dist(early, late)
C /= C.max()

# compute exact OT
a = np.ones(early.shape[0]) / early.shape[0]
b = np.ones(late.shape[0]) / late.shape[0]
P = ot.emd(a, b, C)

# fit nearest neighbor on early clones
nn = NearestNeighbors(n_neighbors=1)
nn.fit(early)

energy = SamplesLoss('energy')
device = 'cuda'

for i in range(lts.test_srcs.shape[0]):
    src = lts.test_srcs[i].numpy()
    # find nearest neighbor in early clones
    _, idx = nn.kneighbors(src.mean(axis=0, keepdims=True))
    
    # find corresponding late clone via OT coupling
    late_idx = np.argmax(P[idx])
    
    # use late states as prediction
    pred = lts.train_tgts[late_idx].numpy()

    energy_error = energy(lts.test_tgts[i].to(device), torch.tensor(pred, dtype=torch.float32).to(device)).item()
    mmd_error = compute_mmd_distance(lts.test_tgts[i].to(device), torch.tensor(pred, dtype=torch.float32).to(device)).item()
    swd_error = compute_sw_distance(lts.test_tgts[i].to(device), torch.tensor(pred, dtype=torch.float32).to(device)).item()

    results['energy'].append(energy_error)
    results['mmd'].append(mmd_error)
    results['swd'].append(swd_error)

In [4]:
results_df = pd.DataFrame(results)
print(results_df.mean())
print(results_df.std()/np.sqrt(len(results_df)))

energy     6.010681
mmd       12.021351
swd        2.184864
dtype: float64
energy    0.164581
mmd       0.329161
swd       0.033205
dtype: float64
